In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = sns.load_dataset('titanic')

In [3]:
df = pd.DataFrame(df[["age","fare","survived"]])

In [4]:
df.head()

,age,fare,survived
0,22.0,7.2500,0
1,38.0,71.2833,1
2,26.0,7.9250,1
3,35.0,53.1000,1
4,35.0,8.0500,0


In [5]:
df.isnull().sum()

age         177
fare          0
survived      0
dtype: int64

In [6]:
df.dropna(inplace=True)

In [7]:
df.shape

(714, 3)

In [8]:
X = df.iloc[:,[0,1]]
Y = df.iloc[:,-1]

In [9]:
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [10]:
clf = DecisionTreeClassifier()

In [11]:
clf.fit(x_train,y_train)
y_pred = clf.predict(x_test)

In [12]:
accuracy_score(y_test,y_pred)

0.6223776223776224

In [13]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,Y,cv=10,scoring='accuracy'))

np.float64(0.6330790297339594)

In [73]:
kbin_age = KBinsDiscretizer(n_bins=25,encode="ordinal",strategy="uniform")
kbin_fare = KBinsDiscretizer(n_bins=25,encode="ordinal",strategy="uniform")

In [74]:
trf = ColumnTransformer([
    ("first",kbin_age,[0]),
    ("second",kbin_fare,[1])
],remainder="passthrough")

In [75]:
x_train_transformed = trf.fit_transform(x_train)
x_test_transformed = trf.transform(x_test)

In [76]:
output = pd.DataFrame({
    'age':x_train["age"],
    'age_trf':x_train_transformed[:,0],
    "fare":x_train['fare'],
    "fare_trf":x_train_transformed[:,1]
})

In [77]:
output["age_label"] = pd.cut(x=x_train['age'],bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_label'] = pd.cut(x = x_train['fare'],bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [78]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_label,fare_label
847,35.0,10.0,7.8958,0.0,"(32.252, 35.435]","(0.0, 20.493]"
554,22.0,6.0,7.7750,0.0,"(19.519, 22.702]","(0.0, 20.493]"
733,23.0,7.0,13.0000,0.0,"(22.702, 25.886]","(0.0, 20.493]"
526,50.0,15.0,10.5000,0.0,"(48.168, 51.351]","(0.0, 20.493]"
395,22.0,6.0,7.7958,0.0,"(19.519, 22.702]","(0.0, 20.493]"


In [79]:
clf = DecisionTreeClassifier()
clf.fit(x_train_transformed,y_train)
y_pred2 = clf.predict(x_test_transformed)

In [80]:
accuracy_score(y_test,y_pred2)

0.6433566433566433